# Day 3 — Hyperparameter Tuning Practice
## Dataset: Loan Approval Prediction

Is notebook mein hum **Hyperparameter Tuning** practice karenge — `loan_approval_data.csv` par, cross-validation ke saath combine kar ke.

**Notebook Structure:**
1. Import Libraries
2. Load Dataset
3. Quick Preprocessing + Feature Engineering (recap)
4. Baseline Model (no tuning)
5. Manual Tuning (concept demo)
6. Grid Search CV
7. Randomized Search CV
8. Comparing Multiple Models with Grid Search
9. Bayesian-style Search (optional, `RandomizedSearchCV` with distributions)
10. Final Model Evaluation
11. Conclusion


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline


## 2. Load Dataset

In [ ]:
df = pd.read_csv('loan_approval_data.csv')
df = df.drop(columns=['Loan_ID'])
df.head()


## 3. Quick Preprocessing + Feature Engineering (Recap)

Yeh wahi steps hain jo Day-1 (Feature Engineering) notebook mein detail se cover kiye the.


In [ ]:
numeric_missing = ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']
categorical_missing = ['Gender', 'Married', 'Dependents', 'Self_Employed']

for col in numeric_missing:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_missing:
    df[col] = df[col].fillna(df[col].mode()[0])

df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Loan_Income_Ratio'] = df['LoanAmount'] / (df['Total_Income'] + 1)
df['EMI'] = df['LoanAmount'] / (df['Loan_Amount_Term'] + 1)
df['Balance_Income'] = df['Total_Income'] - (df['EMI'] * 1000)
df['Dependents_Num'] = df['Dependents'].replace({'3+': '3'}).astype(int)
df = df.drop(columns=['Dependents'])

binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
df = pd.get_dummies(df, columns=['Property_Area'], prefix='Area', drop_first=True)

scale_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Total_Income',
              'EMI', 'Balance_Income', 'Loan_Income_Ratio']
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 4. Baseline Model (No Tuning)

Pehle default hyperparameters ke saath ek Random Forest train karte hain — yeh humara **benchmark** hoga jisse hum tuning ka fayda measure karenge.


In [ ]:
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Baseline Random Forest Accuracy: {baseline_acc:.4f}")
print("\nDefault hyperparameters:")
print(baseline_model.get_params())


## 5. Manual Tuning (Concept Demo)

Tuning shuru karne se pehle, samjhte hain manually — ek hyperparameter (`max_depth`) ko change kar ke performance kaise badalti hai.


In [ ]:
depths = [2, 4, 6, 8, 10, 15, 20, None]
manual_scores = []

for d in depths:
    m = RandomForestClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    manual_scores.append(acc)
    print(f"max_depth={d} -> Accuracy: {acc:.4f}")

plt.figure(figsize=(7, 4))
plt.plot([str(d) for d in depths], manual_scores, marker='o')
plt.title('Effect of max_depth on Accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.show()


**Observation:** Har hyperparameter ko manually try karna time-consuming hai, especially jab multiple hyperparameters ek saath tune karne hon. Isliye hum **Grid Search** aur **Random Search** use karte hain.


## 6. Grid Search CV

**Grid Search** har possible combination of given hyperparameters ko try karta hai, aur cross-validation ke through best combination dhoondta hai.

> **Note:** Grid Search thorough hai lekin slow — combinations exponentially barh jate hain.


In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)


In [ ]:
best_grid_model = grid_search.best_estimator_
grid_test_acc = accuracy_score(y_test, best_grid_model.predict(X_test))

print(f"Baseline Test Accuracy:    {baseline_acc:.4f}")
print(f"Grid Search Test Accuracy: {grid_test_acc:.4f}")


In [ ]:
# Inspect top combinations
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('mean_test_score', ascending=False)
results_df[['params', 'mean_test_score', 'std_test_score']].head(10)


## 7. Randomized Search CV

**Random Search** grid ke saare combinations try nahi karta — balke random sample leta hai. Yeh **faster** hai aur bade parameter spaces ke liye zyada practical, aksar utna hi acha result deta hai jitna Grid Search.


In [ ]:
param_dist = {
    'n_estimators': randint(50, 400),
    'max_depth': randint(3, 25),
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 8),
    'max_features': ['sqrt', 'log2', None],
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=40,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)


In [ ]:
best_random_model = random_search.best_estimator_
random_test_acc = accuracy_score(y_test, best_random_model.predict(X_test))

print(f"Baseline Test Accuracy:        {baseline_acc:.4f}")
print(f"Grid Search Test Accuracy:     {grid_test_acc:.4f}")
print(f"Random Search Test Accuracy:   {random_test_acc:.4f}")


## 8. Comparing Multiple Models with Grid Search

Ab hum tuning ko sirf Random Forest tak mahdood nahi rakhte — Logistic Regression aur Decision Tree ko bhi tune kar ke best overall model dhoondte hain.


In [ ]:
model_grids = {
    'Logistic Regression': (
        LogisticRegression(max_iter=1000),
        {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs']}
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=42),
        {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [100, 200], 'max_depth': [6, 10, None]}
    ),
}

best_models = {}
for name, (estimator, grid) in model_grids.items():
    gs = GridSearchCV(estimator, grid, cv=cv, scoring='accuracy', n_jobs=-1)
    gs.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, gs.predict(X_test))
    best_models[name] = {
        'best_params': gs.best_params_,
        'cv_score': gs.best_score_,
        'test_score': test_acc
    }
    print(f"{name}:")
    print(f"  Best Params: {gs.best_params_}")
    print(f"  CV Accuracy: {gs.best_score_:.4f}  |  Test Accuracy: {test_acc:.4f}\n")


In [ ]:
comparison_df = pd.DataFrame(best_models).T[['cv_score', 'test_score']]
comparison_df.plot(kind='bar', figsize=(8, 5))
plt.title('Tuned Model Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.legend(['CV Accuracy', 'Test Accuracy'])
plt.show()


## 9. Bonus: Continuous Distributions in Randomized Search

`RandomizedSearchCV` sirf lists hi nahi, `scipy.stats` distributions bhi accept karta hai — jaise `uniform` (continuous range) — jo especially Logistic Regression ke `C` parameter jaise continuous hyperparameters ke liye useful hai.


In [ ]:
log_reg_dist = {
    'C': uniform(loc=0.001, scale=100),
    'solver': ['lbfgs', 'liblinear']
}

log_random_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000),
    param_distributions=log_reg_dist,
    n_iter=30,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

log_random_search.fit(X_train, y_train)
print("Best C:", log_random_search.best_params_)
print("Best CV Accuracy:", log_random_search.best_score_)


## 10. Final Model Evaluation

In [ ]:
# Pick whichever tuned model performed best (example: Random Forest from Grid Search)
final_model = best_grid_model
final_pred = final_model.predict(X_test)

print("Final Model Test Accuracy:", accuracy_score(y_test, final_pred))
print("\nClassification Report:\n", classification_report(y_test, final_pred))

cm = confusion_matrix(y_test, final_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Final Tuned Model')
plt.show()


## 11. Conclusion

Is notebook mein humne practice ki:

1. Baseline model (default hyperparameters) banaya — comparison ke liye benchmark
2. Manually ek hyperparameter (`max_depth`) change kar ke uska asar dekha
3. **Grid Search CV** — saare combinations try kar ke best params dhoonde
4. **Randomized Search CV** — random sampling se faster tuning
5. Multiple models (Logistic Regression, Decision Tree, Random Forest) ko tune kar ke compare kiya
6. Continuous distributions (`scipy.stats`) ke saath Randomized Search
7. Final tuned model ka evaluation (classification report + confusion matrix)

**Key takeaway:** Hyperparameter tuning hamesha **cross-validation ke saath** hona chahiye, warna results ek specific split par overfit ho sakte hain.

**Full pipeline recap (Day 1 → 2 → 3):**
`Feature Engineering → Cross-Validation → Hyperparameter Tuning → Final Model`
